# TD0 — les fondamentaux, mis à l'épreuve

**IAA1302.1 — Apprentissage Automatique · ESGC-VAK 2026-2027**

> **Avant ce TD :** `notebook_J0.ipynb` (la syntaxe) et `notebook_J0.5.ipynb` (la chaîne
> complète sur les vraies données).

Ce carnet n'apprend rien de neuf — sauf **une** chose, la fonction `lambda`, qui n'est dans
aucun carnet et qu'on introduit ici. Pour tout le reste, il vous demande de **refaire** ce
que `J0` et `J0.5` ont montré.

## Comment chaque question est faite

Chaque question a **deux cellules**, dans cet ordre :

1. **Ce qu'on montre** — une cellule fournie. Exécutez-la, lisez ce qui s'affiche. Elle
   finit par une ligne `→ …` qui dit ce qu'on vient de comprendre ;
2. **À votre tour** — la même chose, à écrire par vous. Un contrôle vérifie la **forme** de
   votre réponse, et se tait tant que vous n'avez rien écrit.

**Ce que le TD ne fera jamais :** vous donner une valeur. Il dit `ok` ou `REVOIR` ; la valeur,
c'est vous qui la produisez.

## Ce qu'on rend, et où

**Ce carnet, rempli**, déposé dans votre **dépôt GitHub personnel** avec votre déclaration
d'usage de l'IA (`declaration_TD0.md`, recopiée de `DECLARATION_modele.md`). Trois règles :
`Save & Run All` (pas `Save`), carnet **privé**, **CPU, moins de 10 minutes**.

## Les six thèmes, et le temps

| Thème | Questions |
|---|---|
| `pandas` — le tableau, la colonne, lire, calculer, filtrer, grouper, les trous, les fuites | 1 à 8 |
| `numpy` — le vecteur, ses statistiques, le fabriquer | 9 et 10 |
| La compréhension de liste et de dictionnaire | 11 et 12 |
| La fonction `lambda` | 13 et 14 |
| La fonction `def` | 15 et 16 |

**Durée prévue : deux heures et demie.** Ne cherchez pas à tout finir vite : le but est de comprendre
chaque cellule de démonstration avant de passer à la vôtre.

---

## Mise en place (fournie)

Exécutez ces cellules de haut en bas, **sans rien y modifier** : elles posent la mécanique
d'auto-correction, chargent les données, et construisent la petite table des démonstrations.

### Les outils


In [51]:
# les outils du TD — exécutez cette cellule, ne la modifiez pas
# Le TD corrige la forme de vos réponses, jamais leur valeur : il ne vous donne aucune réponse.

# un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

VERIFICATIONS = []


def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de forme, posé à la fin d'une question. Il ne compare jamais à une valeur
       attendue : un type, une plage, une dimension, un compte. Tant que votre réponse n'est
       pas écrite, il se tait — et il le dit.

       `des_que` : la garde d'une réponse qui a une valeur par défaut (une liste vide, par
       exemple). Tant que la garde est vide, le contrôle se tait : sans elle, un cahier
       vierge afficherait des REVOIR que vous ne pouvez pas encore traiter."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))


def bilan():
    """Le compte de vos contrôles : ce qui tient, et ce qu'il reste à revoir."""
    tenus = sum(1 for v in VERIFICATIONS if v['tient'])
    print(f"vos contrôles : {tenus} ok sur {len(VERIFICATIONS)}")
    reste = [v['contrôle'] for v in VERIFICATIONS if not v['tient']]
    if reste:
        print('  à revoir :', reste)

### Les données, et la table des démonstrations


In [52]:
# les données du cours — le TD travaille sur la vraie table de la compétition
# On cherche `train.csv` au lieu de supposer un chemin : Kaggle a déjà déplacé ses points
# de montage ; un chemin recopié se périme, une recherche non.
import os
import pandas as pd
import numpy as np


def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, 'train.csv')):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if 'train.csv' in fichiers:
                return chemin
    return None


DOSSIER = _trouver_dossier(['/kaggle/input', 'sortie',
                            os.path.join('..', 'Donnees_competition', 'sortie')])
assert DOSSIER, 'train.csv introuvable : les données de la compétition sont-elles montées ?'

train = pd.read_csv(os.path.join(DOSSIER, 'train.csv'), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, 'test.csv'), low_memory=False)

ID, CIBLE = 'ligne_id', 'patient_absent'

# la petite table des démonstrations — trois lignes, pour voir chaque geste au clair
voiture = pd.DataFrame({'age': [41, 55, 33],
                        'clinique': ['A1', 'B2', 'A1'],
                        'absent': [0, 1, 1]})

# la table de travail du cours : ce que J0.5 écartait déjà — les questions 7 et 8 s'en servent
A_ECARTER = [ID, 'dossier_id', 'patient_id', 'duree_consultation', 'constantes_relevees',
             'motif_absence', 'date_prise', 'date_rdv', 'taille_cm', 'date_naissance']
X = train.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in train.columns]).copy()
y = train[CIBLE]

In [54]:
# un premier regard
print('train :', train.shape, '(lignes, colonnes)')
print('test  :', test.shape, '(lignes, colonnes)')
print('la cible :', CIBLE)
print('la part d\'absents :', round(100 * train[CIBLE].mean(), 2), '%')

print("la table des démonstrations \n")
display(voiture)

train : (149894, 39) (lignes, colonnes)
test  : (63721, 38) (lignes, colonnes)
la cible : patient_absent
la part d'absents : 18.25 %
la table des démonstrations 



,age,clinique,absent
0,41,A1,0
1,55,B2,1
2,33,A1,1


---

## 1. `pandas` — le tableau et la colonne

C'est l'objet central du cours : un **tableau** à colonnes nommées. Une colonne seule est
une `Series`. Tout le reste du TD travaille sur ces deux objets.

### Question 1 — Construire un tableau, et le lire  · *carnet J0 §10*

**Ce qu'on montre.** Comment on construit un tableau à partir d'un dictionnaire, et les trois
choses qu'on lui demande toujours en premier : ses dimensions, ses colonnes, une colonne seule.

In [55]:
# Démonstration — à exécuter, à lire
brut = {'age': [41, 55, 33], 'clinique': ['A1', 'B2', 'A1']}

tableau = pd.DataFrame(brut)

print('dimensions (lignes, colonnes) :', tableau.shape)
print('noms des colonnes           :', list(tableau.columns))

print("la colonne age, seule \n")
print(tableau['age'])

print('→ un dictionnaire de listes devient un tableau ; une colonne se lit par son nom')

dimensions (lignes, colonnes) : (3, 2)
noms des colonnes           : ['age', 'clinique']
la colonne age, seule 

0    41
1    55
2    33
Name: age, dtype: int64
→ un dictionnaire de listes devient un tableau ; une colonne se lit par son nom


**À votre tour.** Construisez `mon_tableau` depuis le dictionnaire `donnees` ci-dessous
(trois colonnes : `age`, `clinique`, `absent`). Rendez ses dimensions, la liste de ses
colonnes, et la colonne `age` seule.

**Livrables.** `mon_tableau` · `ses_dimensions` · `ses_colonnes` · `la_colonne_age`.

In [57]:
# À vous — écrivez votre réponse ici.

donnees = {'age': [41, 55, 33],
           'clinique': ['A1', 'B2', 'A1'],
           'absent': [0, 1, 1]}

# TODO : le tableau, ses dimensions (.shape), ses colonnes, la colonne age seule.
# mon_tableau = …   ses_dimensions = …   ses_colonnes = …   la_colonne_age = …

controle('trois lignes, trois colonnes', lambda: tuple(ses_dimensions) == (3, 3),
         'un couple (lignes, colonnes)')
controle('les noms des colonnes, dans l’ordre',
         lambda: list(ses_colonnes) == ['age', 'clinique', 'absent'], 'trois noms')
controle('la colonne age seule', lambda: list(la_colonne_age) == [41, 55, 33],
         'les trois âges, dans l’ordre')

  (pas encore écrit)  trois lignes, trois colonnes
  (pas encore écrit)  les noms des colonnes, dans l’ordre
  (pas encore écrit)  la colonne age seule


### Question 2 — Calculer sur une colonne  · *carnet J0 §11*

**Ce qu'on montre.** Une colonne se comporte comme une suite de nombres : les opérations
s'appliquent à toutes ses valeurs **d'un coup**, sans boucle. Et une comparaison rend
`True`/`False` ligne par ligne — qu'on peut compter.

In [ ]:
# Démonstration — à exécuter, à lire
print('moyenne des âges :', voiture['age'].mean())
print('âge maximum      :', voiture['age'].max())
print('combien de 1     :', int(voiture['absent'].sum()))

print("une comparaison, ligne par ligne \n")

print(voiture['age'] > 40)
print('combien de lignes :', int((voiture['age'] > 40).sum()))

print('→ aucune boucle : la colonne entière se calcule en une ligne ; un True vaut 1')

**À votre tour.** Sur `mon_tableau` (que vous venez de construire) : la moyenne des âges,
l'âge maximum, et combien de lignes ont **plus de 40 ans**.

**Livrables.** `age_moyen` · `age_max` · `combien_de_plus_de_40`.

In [59]:
# À vous — écrivez votre réponse ici.

# TODO : trois calculs sur une colonne — aucune boucle nécessaire.
# age_moyen = …   age_max = …   combien_de_plus_de_40 = …

controle('la moyenne des âges', lambda: abs(age_moyen - 43.0) < 0.01, '43.0')
controle('l’âge maximum', lambda: age_max == 55, 'le plus grand des trois')
controle('le compte des plus de 40 ans', lambda: combien_de_plus_de_40 == 2, 'deux lignes')

  (pas encore écrit)  la moyenne des âges
  (pas encore écrit)  l’âge maximum
  (pas encore écrit)  le compte des plus de 40 ans


### Question 3 — Filtrer, et viser une colonne  · *carnet J0 §12*

**Ce qu'on montre.** Trois façons de prendre un morceau de tableau : une condition entre
crochets, `.loc` pour viser une colonne **et** une condition d'un coup, `.iloc` pour viser
par la position. Et la différence entre les trois.

In [60]:
# Démonstration — à exécuter, à lire
print("les absents, avec la condition \n")
print(voiture[voiture['absent'] == 1])
print('dimensions :', voiture[voiture['absent'] == 1].shape)

print("la moyenne d'une colonne visée, sous condition \n")
print('âge moyen des absents :', voiture.loc[voiture['absent'] == 1, 'age'].mean())

print("la première ligne, par sa position \n")
print(voiture.iloc[0])

print('→ la condition garde des lignes ; .loc vise colonne + condition ; .iloc vise par rang')

les absents, avec la condition 

   age clinique  absent
1   55       B2       1
2   33       A1       1
dimensions : (2, 3)
la moyenne d'une colonne visée, sous condition 

âge moyen des absents : 44.0
la première ligne, par sa position 

age         41
clinique    A1
absent       0
Name: 0, dtype: object
→ la condition garde des lignes ; .loc vise colonne + condition ; .iloc vise par rang


**À votre tour.** Sur `mon_tableau` : le tableau des absents (`absent == 1`), sa forme, et
l'âge moyen de **ces seules** lignes.

**Livrables.** `les_absents` · `forme_absents` · `age_moyen_absents`.

In [62]:
# À vous — écrivez votre réponse ici.

# TODO : le filtre, sa forme, et la moyenne visée avec .loc.
# les_absents = …   forme_absents = …   age_moyen_absents = …

controle('le filtre garde deux lignes et toutes les colonnes',
         lambda: tuple(forme_absents) == (2, 3), '(2, 3)')
controle('les deux lignes gardées ont bien absent == 1',
         lambda: int(les_absents['absent'].sum()) == 2, 'la somme de la colonne vaut 2')
controle('la moyenne visée', lambda: abs(age_moyen_absents - 44.0) < 0.01, '44.0')

  (pas encore écrit)  le filtre garde deux lignes et toutes les colonnes
  (pas encore écrit)  les deux lignes gardées ont bien absent == 1
  (pas encore écrit)  la moyenne visée


### Question 4 — Grouper, et trier  · *carnet J1 §30*

**Ce qu'on montre.** `groupby` découpe la table par modalité d'une colonne et calcule un
effectif et un taux par groupe. `sort_values` range le résultat. C'est **le** geste de `J1`.

In [ ]:
# Démonstration — à exécuter, à lire
par_sexe = train.groupby('sexe', observed=True)[CIBLE].agg(
    lignes='size', taux='mean')
par_sexe['taux'] = (100 * par_sexe['taux']).round(2)
par_sexe = par_sexe.sort_values('taux', ascending=False)

print("l'absence, par sexe, du plus fort au plus faible \n")
display(par_sexe)
print('→ .agg compte les lignes et prend la moyenne ; sort_values range le tableau')

**À votre tour.** Refaites le geste pour la colonne `district` : le nombre de lignes et le
taux d'absence en pourcentage, du plus fort au plus faible. Nommez aussi le district au
taux le plus élevé.

**Livrables.** `par_district` · `pire_district`.

In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : groupby + agg(lignes='size', taux='mean'), taux en %, puis sort_values.
# par_district = …   pire_district = …   (l'étiquette du groupe au taux le plus élevé)

controle('un tableau à deux colonnes, six lignes',
         lambda: tuple(par_district.shape) == (6, 2), '(6 modalités, 2 colonnes)')
controle('les colonnes sont nommées',
         lambda: list(par_district.columns) == ['lignes', 'taux'], 'lignes · taux')
controle('le taux est en pourcentage, trié décroissant',
         lambda: 10 < float(par_district['taux'].iloc[0]) < 40
                 and par_district['taux'].iloc[0] >= par_district['taux'].iloc[-1],
         'un pourcentage, du plus fort au plus faible')
controle('une étiquette de groupe est nommée', lambda: pire_district in par_district.index,
         'une des six modalités de district')

### Question 5 — Les trous  · *carnet J1*

**Ce qu'on montre.** Une colonne peut avoir des trous (`NaN`). On les compte avec `isna()`, et
on regarde si le taux d'absence **change** entre les lignes où la colonne manque et celles où
elle est remplie. Deux trous ne racontent pas la même chose.

In [ ]:
# Démonstration — à exécuter, à lire
manque = train['constantes_relevees'].isna()

print('lignes où la colonne manque  :', int(manque.sum()))
print('lignes où elle est remplie   :', int((~manque).sum()))

print("le taux d'absence dans chacun des deux groupes \n")
print('chez les lignes qui manquent :', round(100 * train.loc[manque, CIBLE].mean(), 2), '%')
print('chez les lignes remplies     :', round(100 * train.loc[~manque, CIBLE].mean(), 2), '%')

print('→ isna() donne le masque ; ~masque donne son contraire ; .loc applique le groupe')

**À votre tour.** Faites le même geste sur la colonne **`solde_credit`** : les deux effectifs,
les deux taux en pourcentage, et une phrase qui dit si ce trou raconte quelque chose.

**Livrables.** `n_manque` · `n_rempli` · `taux_manque` · `taux_rempli` · `mon_verdict_trou`.

In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : isna() sur solde_credit, les deux effectifs et les deux taux en %.
# n_manque = …   n_rempli = …   taux_manque = …   taux_rempli = …   mon_verdict_trou = …

controle('les deux effectifs couvrent la table',
         lambda: n_manque + n_rempli == len(train), 'la somme vaut le nombre de lignes')
controle('les deux taux sont des pourcentages',
         lambda: 0 < taux_manque <= 100 and 0 < taux_rempli <= 100,
         'deux pourcentages, pas des proportions')
controle('le verdict est argumenté', lambda: len(str(mon_verdict_trou)) > 40,
         'une phrase : ce trou raconte-t-il quelque chose, ou non ?')

### Question 6 — Un texte qui devrait être un nombre  · *carnet J1 §44*

**Ce qu'on montre.** Le piège le plus courant de la table : un nombre rangé en **texte**, avec
une virgule décimale. Un tri sur du texte range `99` après `100`. La conversion se fait en
deux temps : remplacer la virgule, puis convertir.

In [ ]:
# Démonstration — à exécuter, à lire
print('ce que contient la colonne :', train['taille_cm'].head(3).tolist())
print('son type                  :', train['taille_cm'].dtype)

taille = pd.to_numeric(train['taille_cm'].str.replace(',', '.'), errors='coerce')

print("après conversion \n")
print('type obtenu :', taille.dtype)
print('plus petite :', round(taille.min(), 1), 'cm · plus grande :', round(taille.max(), 1), 'cm')

print('→ str.replace puis to_numeric ; errors="coerce" met NaN sur ce qui ne se convertit pas')

**À votre tour.** Convertissez `taille_cm` en nombre, et rendez le nombre de valeurs
converties et celui des valeurs non convertibles.

**Livrables.** `taille_convertie` · `converties` · `non_convertibles`.

In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : str.replace(',', '.'), puis pd.to_numeric(errors='coerce') ; les deux comptes.
# taille_convertie = …   converties = …   non_convertibles = …

controle('la colonne convertie n’est plus du texte',
         lambda: pd.api.types.is_numeric_dtype(taille_convertie), 'un type numérique')
controle('les deux comptes couvrent la table',
         lambda: converties + non_convertibles == len(train), 'la somme vaut le nombre de lignes')
controle('les deux comptes sont bien mesurés',
         lambda: converties == int(taille_convertie.notna().sum()), 'l’un des deux peut valoir 0')

### Question 7 — Repérer les colonnes, et celles à écarter  · *carnet J0.5 §3*

**Ce qu'on montre.** Un modèle d'arbres accepte les colonnes **catégorielles natives**
jusqu'à **255 modalités** — au-delà, il refuse. On sépare donc les colonnes numériques des
autres, puis on cherche celles qui ont trop de modalités.

In [10]:
# Démonstration — à exécuter, à lire
print('numérique ou non ?')
for c in ['age', 'clinique', 'assurance']:
    print(f'  {c:<12} {pd.api.types.is_numeric_dtype(train[c])}')

print("le nombre de modalités, sur quelques colonnes \n")
for c in ['sexe', 'assurance', 'district', 'quartier']:
    print(f'  {c:<12} {train[c].nunique()} modalités')

print('→ is_numeric_dtype dit le type ; nunique compte les modalités')

numérique ou non ?
  age          True
  clinique     False
  assurance    False
le nombre de modalités, sur quelques colonnes 

  sexe         2 modalités
  assurance    4 modalités
  district     6 modalités
  quartier     80 modalités
→ is_numeric_dtype dit le type ; nunique compte les modalités


In [11]:
for c in ['age', 'clinique', 'assurance']:
    print(f'  {c:<12} {pd.api.types.is_numeric_dtype(train[c])}')
 

  age          True
  clinique     False
  assurance    False


**À votre tour.** Sur la **table de travail `X`** (fournie dans la mise en place : elle a
écarté les identifiants, les dates et les colonnes connues après l'issue), rendez la liste
des colonnes **non numériques**, puis celles qui ont **plus de
255 modalités**, et leur nombre.

**Livrables.** `cat_X` · `cols_trop_fines` · `nombre_trop_fines`.

In [14]:
# À vous — écrivez votre réponse ici.

# TODO : les colonnes non numériques de X (is_numeric_dtype),
#         puis celles à plus de 255 modalités (nunique), et leur nombre.
# cat_X = …   cols_trop_fines = …   nombre_trop_fines = …
import pandas as pd
from pandas.api.types import is_numeric_dtype

# Colonnes non numériques de X
cat_X = [col for col in X.columns if not is_numeric_dtype(X[col])]

# Parmi elles, celles à plus de 255 modalités
cols_trop_fines = [col for col in cat_X if X[col].nunique() > 255]

nombre_trop_fines = len(cols_trop_fines)

print(f"Colonnes non numériques ({len(cat_X)}) : {cat_X}")
print(f"Colonnes à plus de 255 modalités : {cols_trop_fines}")
print(f"Nombre de colonnes trop fines : {nombre_trop_fines}")

print("\nNombre de modalités par colonne non numérique :")
print(X[cat_X].nunique().sort_values(ascending=False))

controle('les colonnes non numériques de X',
         lambda: len(cat_X) == 9, 'neuf colonnes : la table de travail en garde neuf')
controle('une seule colonne dépasse 255 modalités',
         lambda: len(cols_trop_fines) == 1, 'une seule')
controle('la colonne nommée dépasse bien 255 modalités',
         lambda: len(cols_trop_fines) == 1 and X[cols_trop_fines[0]].nunique() > 255,
         'sa cardinalité est supérieure à 255')
controle('le compte suit la liste', lambda: nombre_trop_fines == len(cols_trop_fines),
         'un entier, égal à la longueur de la liste')

Colonnes non numériques (9) : ['clinique', 'district', 'sexe', 'quartier', 'assurance', 'rappel_envoye', 'zone_clinique', 'tournee_jour', 'canal_prise']
Colonnes à plus de 255 modalités : ['tournee_jour']
Nombre de colonnes trop fines : 1

Nombre de modalités par colonne non numérique :
tournee_jour     9000
clinique           80
quartier           80
district            6
assurance           4
canal_prise         4
sexe                2
zone_clinique       1
rappel_envoye       1
dtype: int64
  ok        les colonnes non numériques de X
  ok        une seule colonne dépasse 255 modalités
  ok        la colonne nommée dépasse bien 255 modalités
  ok        le compte suit la liste


### Question 8 — La colonne connue après l'issue  · *carnet J0.5 §3, §18*

**Ce qu'on montre.** Certaines colonnes n'existent **qu'après** l'événement qu'on veut
prédire. Le cours les appelle des **fuites** : elles donnent la réponse. Ici, un motif
d'absence ne peut être écrit que si le patient a manqué son rendez-vous.

In [ ]:
# Démonstration — à exécuter, à lire
print('le taux d’absence, selon que le motif est renseigné ou non \n')
groupe = train['motif_absence'].isna().map({True: 'non renseigné', False: 'renseigné'})
par_motif = train.groupby(groupe, observed=True)[CIBLE].agg(lignes='size', taux='mean')
par_motif['taux'] = (100 * par_motif['taux']).round(2)
display(par_motif)

print('→ quand le motif est renseigné, l’absence est certaine : la colonne donne la réponse')

**À votre tour.** Parmi les colonnes de la liste ci-dessous, **une seule** n'existerait pas
au moment où la décision se prend. Nommez-la, et expliquez en une phrase pourquoi elle ne
peut pas servir de variable d'entrée.

**Livrables.** `ma_fuite` · `pourquoi_ce_nest_pas_une_feature`.

In [18]:
# À vous — écrivez votre réponse ici.

colonnes_a_examiner = ['age', 'sexe', 'assurance', 'delai_jours',
                       'motif_absence', 'canal_prise', 'charge_chronique']

# TODO : laquelle n'existe qu'après la décision ? nommez-la, et dites pourquoi.
# ma_fuite = …   pourquoi_ce_nest_pas_une_feature = …

colonnes_a_examiner = ['age', 'sexe', 'assurance', 'delai_jours',
                       'motif_absence', 'canal_prise', 'charge_chronique']

ma_fuite = 'motif_absence'

pourquoi_ce_nest_pas_une_feature = (
    "motif_absence n'est enregistrée qu'après le rendez-vous, quand le patient "
    "ne s'est pas présenté. Elle n'existe donc pas au moment de la décision. "
    "De plus, elle est vide pour les présents et renseignée pour les absents : "
    "elle révèle directement la cible. Le modèle aurait un score excellent à "
    "l'entraînement, mais serait inutilisable en production, car cette "
    "information serait absente au moment de prédire."
)

features = [c for c in colonnes_a_examiner if c != ma_fuite]

# ---------- Affichage des résultats ----------
print(f"Colonne en fuite : {ma_fuite}")
print(f"\nPourquoi ce n'est pas une feature :\n{pourquoi_ce_nest_pas_une_feature}")
print(f"\nFeatures conservées ({len(features)}) : {features}")
controle('une colonne de la liste est nommée',
         lambda: ma_fuite in colonnes_a_examiner,
         'le nom exact d’une des sept colonnes à examiner')
controle('la raison est argumentée', lambda: len(str(pourquoi_ce_nest_pas_une_feature)) > 40,
         'une phrase : quand la colonne devient disponible, par rapport à la décision')

Colonne en fuite : motif_absence

Pourquoi ce n'est pas une feature :
motif_absence n'est enregistrée qu'après le rendez-vous, quand le patient ne s'est pas présenté. Elle n'existe donc pas au moment de la décision. De plus, elle est vide pour les présents et renseignée pour les absents : elle révèle directement la cible. Le modèle aurait un score excellent à l'entraînement, mais serait inutilisable en production, car cette information serait absente au moment de prédire.

Features conservées (6) : ['age', 'sexe', 'assurance', 'delai_jours', 'canal_prise', 'charge_chronique']
  ok        une colonne de la liste est nommée
  ok        la raison est argumentée


---

## 2. `numpy` — le vecteur et ses statistiques

`numpy` entre dans le cours à la séance 3, pour trois gestes précis : un **vecteur** de
nombres, une **moyenne**, un **écart-type**. Une colonne de `pandas` se convertit en vecteur
`numpy` quand on le demande.

### Question 9 — Le vecteur, et ses statistiques  · *carnet J3 §26*

**Ce qu'on montre.** Un vecteur `numpy` est une suite de nombres sur laquelle on calcule
directement. `np.std(..., ddof=1)` est l'écart-type **d'échantillon** — celui du cours.

In [38]:
# Démonstration — à exécuter, à lire
notes = np.array([82.5, 82.4, 82.6, 82.51, 82.55])

print('le vecteur     :', notes)
print('sa moyenne     :', round(float(np.mean(notes)), 4))
print('son écart-type :', round(float(np.std(notes, ddof=1)), 4))

print("le vecteur est une suite de nombres, pas une colonne \n")
print('sa forme       :', notes.shape)

print('→ np.array range une suite ; np.mean et np.std(ddof=1) la résument')

le vecteur     : [82.5  82.4  82.6  82.51 82.55]
sa moyenne     : 82.512
son écart-type : 0.074
le vecteur est une suite de nombres, pas une colonne 

sa forme       : (5,)
→ np.array range une suite ; np.mean et np.std(ddof=1) la résument


**À votre tour.** Construisez `mon_vecteur` depuis la liste `graines` ci-dessous, puis rendez
sa moyenne et son écart-type d'échantillon.

**Livrables.** `mon_vecteur` · `moyenne_np` · `ecart_type_np`.

### Question 10 — Fabriquer un vecteur  · *carnet J3 §23*

**Ce qu'on montre.** Trois façons de fabriquer un vecteur sans partir d'une liste : des zéros,
une suite régulière (`arange`), et les valeurs distinctes d'une suite (`unique`).

In [24]:
# À vous — écrivez votre réponse ici.

graines = [82.5, 82.4, 82.6, 82.51, 82.55]

# TODO : le vecteur, sa moyenne, son écart-type (ddof=1).
# mon_vecteur = …   moyenne_np = …   ecart_type_np = …
import numpy as np

graines = [82.5, 82.4, 82.6, 82.51, 82.55]

mon_vecteur = np.array(graines)
moyenne_np = np.mean(mon_vecteur)
ecart_type_np = np.std(mon_vecteur, ddof=1)

print(f"Vecteur : {mon_vecteur}")
print(f"Moyenne : {moyenne_np:.4f}")
print(f"Écart-type (ddof=1) : {ecart_type_np:.4f}")

controle('un vecteur numpy de cinq valeurs',
         lambda: isinstance(mon_vecteur, np.ndarray) and len(mon_vecteur) == 5,
         'un objet numpy, pas une liste')
controle('la moyenne et l’écart-type',
         lambda: abs(moyenne_np - 82.512) < 0.01 and 0 < ecart_type_np < 0.5,
         'deux nombres : la moyenne, puis la dispersion')

Vecteur : [82.5  82.4  82.6  82.51 82.55]
Moyenne : 82.5120
Écart-type (ddof=1) : 0.0740
  ok        un vecteur numpy de cinq valeurs
  ok        la moyenne et l’écart-type


In [39]:
# Démonstration — à exécuter, à lire
print('des zéros, aussi long que la cible :', np.zeros(4, dtype=int))
print('une suite régulière (0 à 10, pas 2) :', np.arange(0, 10, 2))
print('les valeurs distinctes              :', np.unique([3, 1, 3, 2, 1]))

print("la longueur du vecteur de zéros suit celle qu'on lui donne \n")
print('longueur de train :', len(train), '· un vecteur aussi long :', len(np.zeros(len(train), dtype=int)))

print('→ zeros/arange fabriquent un vecteur ; unique range les valeurs une seule fois')

des zéros, aussi long que la cible : [0 0 0 0]
une suite régulière (0 à 10, pas 2) : [0 2 4 6 8]
les valeurs distinctes              : [1 2 3]
la longueur du vecteur de zéros suit celle qu'on lui donne 

longueur de train : 149894 · un vecteur aussi long : 149894
→ zeros/arange fabriquent un vecteur ; unique range les valeurs une seule fois


**À votre tour.** Fabriquez `zeros_de_y`, un vecteur de zéros **aussi long que `y`** (la
cible), en entiers. Rendez aussi ses deux valeurs distinctes.

**Livrables.** `zeros_de_y` · `valeurs_distinctes`.

In [26]:
# À vous — écrivez votre réponse ici.

y = train[CIBLE]

# TODO : np.zeros(len(y), dtype=int), puis np.unique dessus.
# zeros_de_y = …   valeurs_distinctes = …
import numpy as np

y = train[CIBLE]

zeros_de_y = np.zeros(len(y), dtype=int)
valeurs_distinctes = np.unique(zeros_de_y
print(f"Longueur de zeros_de_y : {len(zeros_de_y)}")
print(f"Aperçu : {zeros_de_y[:10]}")
print(f"Valeurs distinctes : {valeurs_distinctes}")

controle('le vecteur est aussi long que la cible', lambda: len(zeros_de_y) == len(y),
         'la longueur de y')
controle('il est entièrement nul, et en entiers',
         lambda: int(zeros_de_y.sum()) == 0 and 'int' in str(zeros_de_y.dtype),
         'la somme vaut 0, le type est entier')
controle('une seule valeur distincte', lambda: len(valeurs_distinctes) == 1, 'un vecteur constant')

Longueur de zeros_de_y : 149894
Aperçu : [0 0 0 0 0 0 0 0 0 0]
Valeurs distinctes : [0]
  ok        le vecteur est aussi long que la cible
  ok        il est entièrement nul, et en entiers
  ok        une seule valeur distincte


---

## 3. La compréhension — construire une suite en une ligne

Cette écriture est partout dans les carnets du cours. Elle veut dire : *« construis la liste
des `c` tels que … »*. On l'écrit pour une liste, et pour un dictionnaire.

### Question 11 — La compréhension de liste  · *carnet J0 §7*

**Ce qu'on montre.** Une compréhension transforme chaque élément, et peut **filtrer** avec un
`if` sur la même ligne. C'est le geste le plus fréquent des carnets du cours.

In [27]:
# Démonstration — à exécuter, à lire
colonnes = ['clinique', 'district', 'sexe', 'assurance']

print('en majuscules :', [c.upper() for c in colonnes])
print('leurs longueurs :', [len(c) for c in colonnes])

print("avec une condition \n")
print('celles qui commencent par s :', [c for c in colonnes if c.startswith('s')])

print('→ on lit la transformation d’abord, la condition à la fin')

en majuscules : ['CLINIQUE', 'DISTRICT', 'SEXE', 'ASSURANCE']
leurs longueurs : [8, 8, 4, 9]
avec une condition 

celles qui commencent par s : ['sexe']
→ on lit la transformation d’abord, la condition à la fin


**À votre tour.** En **une ligne chacune**, rendez les longueurs de toutes les colonnes, puis
les colonnes qui commencent par la lettre `d`.

**Livrables.** `mes_longueurs` · `en_d`.

In [34]:
# À vous — écrivez votre réponse ici.

# TODO : une compréhension pour les longueurs, une avec condition pour le début.
# mes_longueurs = …   en_# Je suppose que la liste de départ s'appelle `mots` (liste de chaînes).
# Remplacez le nom si votre énoncé en utilise un autre.

mots = ["donnée", "modèle", "dataset", "feature", "décision"]

mes_longueurs = [len(m) for m in mots]

en_d = [m for m in mots if m.startswith("d")]

print(f"Mots : {mots}")
print(f"Longueurs : {mes_longueurs}")
print(f"Mots commençant par 'd' : {en_d}")

controle('quatre longueurs, dans l’ordre',
         lambda: list(mes_longueurs) == [8, 8, 4, 9], 'quatre nombres')
controle('la seule colonne qui commence par d', lambda: list(en_d) == ['district'],
         'la condition porte sur le nom (startswith)')

Mots : ['donnée', 'modèle', 'dataset', 'feature', 'décision']
Longueurs : [6, 6, 7, 7, 8]
Mots commençant par 'd' : ['donnée', 'dataset', 'décision']
  REVOIR    quatre longueurs, dans l’ordre   ->   quatre nombres
  REVOIR    la seule colonne qui commence par d   ->   la condition porte sur le nom (startswith)


### Question 12 — La compréhension de dictionnaire  · *carnet J0 §7*

**Ce qu'on montre.** La même idée pour un **dictionnaire** : `{clé: valeur pour chaque élément}`.
C'est ainsi qu'on associe une colonne à son rang, ou un nom à un objet.

In [35]:
# Démonstration — à exécuter, à lire
colonnes = ['clinique', 'district', 'sexe', 'assurance']

rangs = {c: i for i, c in enumerate(colonnes)}
print('chaque colonne et son rang :', rangs)

print("on peut aussi associer un nom à sa longueur \n")
longueurs = {c: len(c) for c in colonnes}
print(longueurs)

print('→ chaque élément produit une paire clé-valeur, sur une seule ligne')

chaque colonne et son rang : {'clinique': 0, 'district': 1, 'sexe': 2, 'assurance': 3}
on peut aussi associer un nom à sa longueur 

{'clinique': 8, 'district': 8, 'sexe': 4, 'assurance': 9}
→ chaque élément produit une paire clé-valeur, sur une seule ligne


**À votre tour.** Construisez `mon_parcours`, le dictionnaire qui associe chaque colonne à
son rang, **sur les trois premières** seulement.

**Livrable.** `mon_parcours`.

In [37]:
# À vous — écrivez votre réponse ici.

# TODO : un dictionnaire {colonne: rang} sur les trois premières — `enumerate` donne le rang.
# mon_parcours 

mon_parcours = {col: rang for rang, col in enumerate(colonnes[:3])}
print(mon_parcours)

controle('trois colonnes, chacune avec son rang',
         lambda: mon_parcours == {'clinique': 0, 'district': 1, 'sexe': 2},
         'la colonne et son rang, à partir de 0')

{'clinique': 0, 'district': 1, 'sexe': 2}
  ok        trois colonnes, chacune avec son rang


---

## 4. La fonction `lambda` — une fonction sans nom

**Cette notion n'est dans aucun carnet du cours : on l'introduit ici.**

Une `lambda` est une fonction écrite **en une ligne**, et **sans nom**.
Elle sert quand on veut dire *« trie selon ce calcul »* ou *« transforme chaque valeur de
cette façon »*, sans écrire une fonction complète avec `def`.

```python
lambda x: x + 1          # se lit : « reçois x, rends x + 1 »
```

Le mot `lambda` remplace `def`, et le nom de la fonction. Le `:` remplace le `return`.
On l'utilise **à l'endroit où on en a besoin** — jamais rangée dans une variable, presque jamais.

### Question 13 — Trier et filtrer par un critère  · *notion nouvelle*

**Ce qu'on montre.** `sorted(..., key=...)` range selon un **critère** calculé. `filter(...)`
garde les éléments qui passent un test. Dans les deux cas, la `lambda` est le critère.

In [ ]:
# Démonstration — à exécuter, à lire
notes = [12, 7, 15, 9]
mots = ['clinique', 'sexe', 'age', 'assurance']

print('ordre normal            :', sorted(notes))
print('du plus grand au plus petit :', sorted(notes, key=lambda n: -n))
print('rangés par longueur      :', sorted(mots, key=lambda m: len(m)))

print("garder selon un test \n")
print('les nombres > 10 :', list(filter(lambda n: n > 10, notes)))

print('→ la lambda dit le critère ; sorted et filter l’appliquent')

**À votre tour.** Rangez `mes_mots` du **plus long au plus court**, puis gardez seulement les
mots de plus de 4 lettres.

**Livrables.** `mots_tries` · `mots_longs`.

In [ ]:
# À vous — écrivez votre réponse ici.

mes_mots = ['age', 'assurance', 'sexe', 'clinique', 'district']

# TODO : sorted avec une lambda, puis filter avec une lambda.
# mots_tries = …   mots_longs = …

controle('les mots sont rangés du plus long au plus court',
         lambda: [len(m) for m in mots_tries] == sorted([len(m) for m in mes_mots], reverse=True),
         'le premier est le plus long')
controle('tous les mots gardés ont plus de quatre lettres',
         lambda: all(len(m) > 4 for m in list(mots_longs)), 'un critère sur la longueur')
controle('le filtre a bien retiré quelque chose',
         lambda: len(list(mots_longs)) < len(mes_mots), 'il en reste moins que le départ')

### Question 14 — `lambda` dans `pandas`  · *notion nouvelle*

**Ce qu'on montre.** Dans `pandas`, `apply(lambda ...)` transforme **chaque valeur** d'une
colonne. C'est ainsi qu'on fabrique une colonne de 0/1 à partir d'une condition sur une autre.

In [ ]:
# Démonstration — à exécuter, à lire
print('chaque âge transformé :', voiture['age'].apply(lambda v: v * 2).tolist())

print("une colonne de test, puis son compte \n")
grand = voiture['age'].apply(lambda v: v > 40)
print(grand.tolist())
print('combien :', int(grand.sum()))

print("une transformation de texte \n")
print(voiture['clinique'].apply(lambda s: s.lower()).tolist())

print('→ apply parcourt la colonne valeur par valeur, et assemble le résultat')

**À votre tour.** Avec `apply(lambda ...)`, fabriquez la colonne `majuscule`, qui met la
colonne `clinique` de `mon_tableau` en minuscules, et comptez combien de lignes valent `'a1'`.

**Livrables.** `majuscule` · `combien_a1`.

In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : mon_tableau['clinique'].apply(lambda s: s.lower()), puis le compte des 'a1'.
# majuscule = …   combien_a1 = …

controle('une colonne transformée, aussi longue que la table',
         lambda: len(majuscule) == len(mon_tableau), 'une valeur par ligne')
controle('la transformation a bien eu lieu',
         lambda: all(s == s.lower() for s in list(majuscule)),
         'chaque valeur est en minuscules')
controle('le compte porte sur la colonne transformée',
         lambda: combien_a1 == int((majuscule == 'a1').sum()), 'un entier')

---

## 5. La fonction `def` — nommer un morceau de code

Dans ce cours, **tout ce que vous rendez doit pouvoir être appelé par quelqu'un d'autre** :
c'est pour ça qu'on écrit des fonctions, et pas seulement des cellules qui s'enchaînent.

La différence avec `lambda` tient en une ligne : une `lambda` n'a **pas de nom**.
Elle sert de critère à une autre fonction. Une `def` **porte un nom**, et on la rappelle tant
qu'on veut.

### Question 15 — Écrire une fonction, avec sa docstring  · *carnet J0 §9*

**Ce qu'on montre.** Une fonction se définit avec `def`, porte une **docstring** (la ligne
entre triples guillemets, qui dit ce qu'elle fait), et rend son résultat avec `return`.
Elle peut rendre **deux** choses à la fois.

In [ ]:
# Démonstration — à exécuter, à lire
def resume(valeurs):
    """Renvoie le nombre de valeurs, et la plus grande."""
    return len(valeurs), max(valeurs)

print('la docstring :', resume.__doc__)
print('appel simple :', resume([10, 12, 14]))

print("deux valeurs rendues se récupèrent sur une ligne \n")
combien, plus_grand = resume([10, 12, 14])
print('le compte :', combien, '· le plus grand :', plus_grand)

print('→ def nomme le code, la docstring l’explique, return le rend')

**À votre tour.** Écrivez `moyenne(liste)` avec sa **docstring**, qui rend la moyenne d'une
liste de nombres. Appelez-la sur `notes`, et rendez aussi sa docstring.

**Livrables.** `moyenne` (la fonction) · `ma_moyenne` · `sa_docstring`.

In [ ]:
# À vous — écrivez votre réponse ici.

notes = [10, 12, 14, 16]

# TODO : une fonction `moyenne` avec docstring, puis l'appel et la lecture de la docstring.
# def moyenne(liste):
#     """…"""
#     return …
# ma_moyenne = …   sa_docstring = …

controle('la fonction porte une docstring',
         lambda: moyenne.__doc__ is not None and len(moyenne.__doc__.strip()) > 10,
         'au moins dix caractères, entre triples guillemets')
controle('elle rend bien la moyenne', lambda: abs(ma_moyenne - 13.0) < 0.01, '13.0')
controle('la docstring a été lue', lambda: len(str(sa_docstring)) > 10, 'le texte de la docstring')

### Question 16 — Le seuil par défaut, et la chaîne complète  · *carnet J1*

**Ce qu'on montre.** Une fonction peut avoir un **argument avec une valeur par défaut** : on
peut l'appeler sans le préciser, ou en le changeant. Ensuite, le geste final du cours :
entraîner un modèle et le mesurer sur des données qu'il n'a pas vues.

In [ ]:
# Démonstration — à exécuter, à lire
def taux_alerte(df, seuil=0.5):
    """Part des lignes dont `taux_absence_passe` dépasse `seuil`."""
    return float((df['taux_absence_passe'] > seuil).mean())

print('sans préciser le seuil :', round(taux_alerte(train), 5))
print('avec un seuil de 0.0   :', round(taux_alerte(train, 0.0), 5))

print("le geste final : entraîner, puis mesurer \n")
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier

# `X` est fourni dans la mise en place ; on écarte seulement les colonnes trop fines
CAT = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
trop_fin = [c for c in CAT if X[c].nunique() > 255]
X = X.drop(columns=trop_fin)
for c in [c for c in CAT if c not in trop_fin]:
    X[c] = X[c].fillna('').astype('category')

Xa, Xc, ya, yc = train_test_split(X, train[CIBLE], test_size=0.2, stratify=train[CIBLE], random_state=0)
modele = HistGradientBoostingClassifier(random_state=0).fit(Xa, ya)
print('exactitude de validation :', round(100 * modele.score(Xc, yc), 2), '%')
print('la référence constante   :', round(100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean()), 2), '%')

print('→ un argument par défaut rend la fonction appelable sans le préciser')

**À votre tour.** Écrivez `part_absents(df)`, qui rend la part d'absents de la table en
**pourcentage**, arrondie à deux décimales. Appelez-la sur `train`.

**Livrables.** `part_absents` (la fonction) · `ma_part_absents`.

In [ ]:
# À vous — écrivez votre réponse ici.

# TODO : def part_absents(df) — part d'absents de df[CIBLE], en %, arrondie.
# def part_absents(df):
#     """…"""
#     return …
# ma_part_absents = …

controle('la fonction est écrite avec sa docstring',
         lambda: part_absents.__doc__ is not None and len(part_absents.__doc__.strip()) > 10,
         'une docstring')
controle('elle rend un pourcentage', lambda: 15 < ma_part_absents < 22,
         'un pourcentage entre 15 et 22')
controle('elle s’applique à une autre table',
         lambda: 0 <= part_absents(train.head(100)) <= 100,
         'elle reçoit une table en argument')

---

## 6. Le bilan

Cette cellule compte vos contrôles. Un contrôle qui affiche `(pas encore écrit)` ou `REVOIR`
est à reprendre. Le TD est réussi quand **tout est `ok`**.

In [ ]:
bilan()

## 7. Rendre le travail

1. **Exécutez tout** le carnet (`Save & Run All`), pour que les contrôles soient passés.
2. **Déposez** `TD0.ipynb` dans votre **dépôt GitHub personnel**, avec votre déclaration
   d'usage de l'IA : recopiez `DECLARATION_modele.md` sous le nom `declaration_TD0.md`,
   remplissez-la, et committez **les deux fichiers dans le même commit**.
3. Vérifiez que le lien de votre dépôt est à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.

---

### Ce que ce TD a mis à l'épreuve

| Thème | Questions | Ce qu'il faut savoir faire |
|---|---|---|
| `pandas` | 1 à 8 | construire et lire un tableau, calculer, filtrer, grouper, voir un trou, convertir du texte, écarter les colonnes, repérer une fuite |
| `numpy` | 9 et 10 | un vecteur, sa moyenne et son écart-type, le fabriquer |
| Compréhensions | 11 et 12 | transformer et filtrer en une ligne, pour une liste et un dictionnaire |
| `lambda` | 13 et 14 | un critère d'une ligne : trier, filtrer, transformer une colonne |
| `def` | 15 et 16 | nommer une fonction, sa docstring, un argument par défaut |

**Si un thème vous a résisté, c'est le carnet correspondant qu'il faut relire** — et c'est
exactement ce que le TD sert à savoir.